# Modelo genérico · Rede Neural (MLP) multiclasse com softmax

**Template reutilizável** para classificar em **3 ou mais classes**: ReLU nas camadas ocultas, saída linear (*logits*) e softmax acoplada à perda (`from_logits=True`), com uma implementação NumPy **numericamente estável** da softmax.

| | |
|---|---|
| **Modelo no sistema** | Machine Learning › 5. Redes neurais e Deep Learning › cartão *Rede Neural (MLP)* (`/modelar/redes-neurais-e-deep-learning/#m-mlp`) |
| **Origem** | `C2_W2_Assignment.ipynb` (*Machine Learning Specialization*, DeepLearning.AI e Stanford Online; reconhecimento dos dígitos 0 a 9 em imagens 20×20), revisado, corrigido e generalizado |
| **Tipo de problema** | Supervisionado · classificação multiclasse |
| **Dependências** | `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `keras` 3 |

### Como usar
1. Edite apenas a célula **1. Configuração** (`>>> ADAPTAR`).
2. Com `CAMINHO_DADOS = None`, roda com dados sintéticos de 4 classes.
3. O alvo pode ter rótulos de texto ou números quaisquer: são convertidos para `0 … K−1` e de volta.
4. Se cada linha for uma imagem achatada, informe `FORMATO_IMAGEM = (altura, largura)`.

### O que mudou em relação ao notebook original
- `my_softmax` calculava `exp(z)` direto: com logits grandes (≈ 710 ou mais) dá *overflow* e devolve `nan`. Agora subtrai `max(z)` antes da exponencial (resultado matematicamente idêntico) e aceita lotes `(m, N)`.
- O original treinava e contava erros **no próprio treino** (5000 imagens). Entraram treino/validação/teste estratificados, *early stopping* e as métricas no teste (acurácia, F1 macro, log loss, matriz de confusão por classe).
- O número de classes deixou de ser fixo em 10 (`Dense(10)`): vem do alvo. Os rótulos são codificados com `LabelEncoder`, então classes de texto funcionam.
- Dependências do curso removidas: `autils.py`, `lab_utils_softmax.py`, `public_tests.py`, o estilo `deeplearning.mplstyle` e o modo `%matplotlib widget` (que exige `ipympl`). Os gráficos agora são estáticos.
- Keras 3 (`import keras`) e `model.save(".keras")`, como no cartão.

## 1. Configuração

In [ ]:
# >>> ADAPTAR: única célula que precisa ser editada para usar outro dataset.

# --- dados ---
CAMINHO_DADOS = None        # ex.: "dados/produtos.csv"; None = dados sintéticos
SEPARADOR = None
TEM_CABECALHO = True        # False se a primeira linha já é dado
NOMES_COLUNAS = None        # nomes quando TEM_CABECALHO = False; sem nomes, viram coluna_0, coluna_1, …
COLUNA_ALVO = "alvo"        # coluna com 3 ou mais classes
COLUNAS_FEATURES = None     # None = todas as demais
FORMATO_IMAGEM = None       # ex.: (20, 20) se cada linha for uma imagem achatada
TRANSPOR_IMAGEM = False     # True se os pixels estão em ordem de coluna (MATLAB/Octave), como no dataset original

# --- divisão e reprodutibilidade ---
TAMANHO_TESTE = 0.2
TAMANHO_VALIDACAO = 0.2
SEMENTE = 1234

# --- rede ---
CAMADAS_OCULTAS = [25, 15]  # como no original
TAXA_APRENDIZADO = 0.001
EPOCAS = 200
TAMANHO_LOTE = 32
PACIENCIA = 15              # None = sem early stopping

In [ ]:
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from keras import layers
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

keras.utils.set_random_seed(SEMENTE)
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## 2. Ativação ReLU e função softmax
A ReLU, $a = \max(0, z)$, é linear quando ligada e zero quando desligada; essa faixa "desligada" é o que a torna não linear e permite que várias unidades contribuam sem interferir umas nas outras. A sigmoide é melhor para saídas liga/desliga; a ReLU, para ativações ocultas com faixa contínua.

A softmax converte o vetor $\mathbf{z}$ de $N$ saídas em uma distribuição de probabilidade:

$$a_j = \frac{e^{z_j}}{\sum_{k=0}^{N-1} e^{z_k}}$$

Como $\frac{e^{z_j - c}}{\sum_k e^{z_k - c}} = \frac{e^{z_j}}{\sum_k e^{z_k}}$ para qualquer $c$, usar $c = \max(\mathbf{z})$ dá o mesmo resultado sem *overflow*.

In [ ]:
z = np.linspace(-3, 3, 200)
fig, ax = plt.subplots(1, 3, figsize=(11, 2.8))
for eixo, (nome, f) in zip(ax, [("linear", z), ("sigmoide", 1 / (1 + np.exp(-z))), ("ReLU", np.maximum(0, z))]):
    eixo.plot(z, f); eixo.set_title(nome); eixo.axhline(0, lw=0.5); eixo.axvline(0, lw=0.5)
plt.tight_layout(); plt.show()

In [ ]:
def my_softmax(z):
    """Softmax estável. z: (N,) ou (m, N) → probabilidades na última dimensão (somam 1)."""
    z = np.asarray(z, dtype=float)
    ez = np.exp(z - np.max(z, axis=-1, keepdims=True))
    return ez / np.sum(ez, axis=-1, keepdims=True)


def softmax_ingenua(z):          # versão do original, só para comparação
    ez = np.exp(z)
    return ez / np.sum(ez)


z = np.array([1., 2., 3., 4.])
print("my_softmax(z):     ", my_softmax(z).round(4))
print("keras softmax(z):  ", keras.ops.convert_to_numpy(keras.ops.softmax(z)).round(4))
assert np.allclose(my_softmax(z), [0.0320586, 0.08714432, 0.23688282, 0.64391426])

grande = np.array([1000., 1001., 1002.])
with np.errstate(over="ignore", invalid="ignore"):
    print("ingênua com logits grandes:", softmax_ingenua(grande), "→ overflow")
print("estável com logits grandes:", my_softmax(grande).round(4))

## 3. Carregamento dos dados

In [ ]:
def carregar_tabela(caminho, separador=None, tem_cabecalho=True, nomes_colunas=None, planilha=0):
    """Lê CSV/TXT/TSV, Excel, Parquet ou JSON e devolve um DataFrame."""
    caminho = Path(caminho)
    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho.resolve()}")
    ext = caminho.suffix.lower()
    cabecalho = 0 if tem_cabecalho else None
    if ext in {".csv", ".txt", ".tsv"}:
        sep = separador if separador is not None else ("\t" if ext == ".tsv" else None)
        df = pd.read_csv(caminho, sep=sep, engine="python", header=cabecalho)
    elif ext in {".xlsx", ".xls"}:
        df = pd.read_excel(caminho, sheet_name=planilha, header=cabecalho)
    elif ext == ".parquet":
        df = pd.read_parquet(caminho)
    elif ext == ".json":
        df = pd.read_json(caminho)
    else:
        raise ValueError(f"Formato não suportado: {ext}")
    if nomes_colunas is not None:
        df.columns = nomes_colunas
    elif not tem_cabecalho:
        df.columns = [f"coluna_{i}" for i in range(df.shape[1])]
    return df


def dados_sinteticos(n=2000, semente=SEMENTE):
    from sklearn.datasets import make_classification
    X, y = make_classification(n_samples=n, n_features=10, n_informative=6, n_classes=4,
                               n_clusters_per_class=1, class_sep=1.2, random_state=semente)
    df = pd.DataFrame(X, columns=[f"feature_{i + 1}" for i in range(X.shape[1])])
    df["alvo"] = np.array(["classe_a", "classe_b", "classe_c", "classe_d"])[y]
    return df


df = (dados_sinteticos() if CAMINHO_DADOS is None
      else carregar_tabela(CAMINHO_DADOS, SEPARADOR, TEM_CABECALHO, NOMES_COLUNAS))
if COLUNA_ALVO not in df.columns:
    raise KeyError(f"COLUNA_ALVO={COLUNA_ALVO!r} não existe. Colunas: {list(df.columns)}")
df = df.dropna(subset=[COLUNA_ALVO])

codificador = LabelEncoder()
y = codificador.fit_transform(df[COLUNA_ALVO])
classes = codificador.classes_
K = len(classes)
if K < 3:
    print("ATENÇÃO: só 2 classes; o template binário (saída sigmoide) é a escolha natural.")
features = COLUNAS_FEATURES or [c for c in df.columns if c != COLUNA_ALVO]
X_df = df[features]
print("formato:", df.shape, "| classes:", K)
display(df[COLUNA_ALVO].value_counts().to_frame("exemplos"))
if FORMATO_IMAGEM is not None and np.prod(FORMATO_IMAGEM) != X_df.shape[1]:
    raise ValueError(f"FORMATO_IMAGEM={FORMATO_IMAGEM} não corresponde a {X_df.shape[1]} features.")

In [ ]:
def como_imagem(linha):
    img = np.asarray(linha, dtype=float).reshape(FORMATO_IMAGEM)
    return img.T if TRANSPOR_IMAGEM else img


def grade_imagens(X, titulos, n=64, semente=SEMENTE, titulo=""):
    """Mostra n linhas como imagens (só quando FORMATO_IMAGEM está definido)."""
    if FORMATO_IMAGEM is None:
        return
    rng = np.random.default_rng(semente)
    idx = rng.choice(len(X), size=min(n, len(X)), replace=False)
    lado = int(np.ceil(np.sqrt(len(idx))))
    fig, eixos = plt.subplots(lado, lado, figsize=(6, 6))
    for ax in eixos.flat:
        ax.set_axis_off()
    for ax, i in zip(eixos.flat, idx):
        ax.imshow(como_imagem(X[i]), cmap="gray")
        ax.set_title(titulos[i], fontsize=8)
    fig.suptitle(titulo)
    plt.tight_layout()
    plt.show()


grade_imagens(X_df.to_numpy(), [str(v) for v in df[COLUNA_ALVO]], titulo="rótulo")

## 4. Preparação

In [ ]:
def montar_preprocessador(X, padronizar=True, remover_primeira=False):
    """Numéricas: mediana + padronização. Categóricas: moda + one-hot.
    remover_primeira=True descarta uma categoria por variável (evita a colinearidade perfeita
    com o intercepto em modelos lineares). Ajuste SÓ no treino (evita vazamento do teste)."""
    numericas = X.select_dtypes(include="number").columns.tolist()
    categoricas = [c for c in X.columns if c not in numericas]
    passos_num = [SimpleImputer(strategy="median")] + ([StandardScaler()] if padronizar else [])
    transformadores = []
    if numericas:
        transformadores.append(("num", make_pipeline(*passos_num), numericas))
    if categoricas:
        transformadores.append(("cat", make_pipeline(
            SimpleImputer(strategy="most_frequent"),
            OneHotEncoder(handle_unknown="ignore", sparse_output=False,
                          drop="first" if remover_primeira else None)), categoricas))
    return ColumnTransformer(transformadores, verbose_feature_names_out=False)


X_rest, X_te_df, y_rest, y_test = train_test_split(X_df, y, test_size=TAMANHO_TESTE, random_state=SEMENTE, stratify=y)
X_tr_df, X_va_df, y_train, y_val = train_test_split(X_rest, y_rest, test_size=TAMANHO_VALIDACAO,
                                                    random_state=SEMENTE, stratify=y_rest)
prep = montar_preprocessador(X_tr_df)
X_train = prep.fit_transform(X_tr_df).astype("float32")
X_val = prep.transform(X_va_df).astype("float32")
X_test = prep.transform(X_te_df).astype("float32")
n_features = X_train.shape[1]
print(f"treino {X_train.shape} | validação {X_val.shape} | teste {X_test.shape}")

## 5. Modelo
Camadas ocultas com ReLU e uma camada de saída com **K unidades lineares**. A softmax não fica na última camada: ela é aplicada dentro da perda (`SparseCategoricalCrossentropy(from_logits=True)`), o que é numericamente mais estável. Consequência na hora de usar: `model.predict` devolve *logits*, e a softmax precisa ser aplicada se você quiser probabilidades. O alvo continua sendo o índice da classe (`0 … K−1`), sem one-hot.

In [ ]:
model = keras.Sequential(
    [keras.Input(shape=(n_features,))]
    + [layers.Dense(u, activation="relu", name=f"L{i + 1}") for i, u in enumerate(CAMADAS_OCULTAS)]
    + [layers.Dense(K, activation="linear", name=f"L{len(CAMADAS_OCULTAS) + 1}")],
    name="mlp_multiclasse",
)
model.summary()
tamanhos = [n_features] + CAMADAS_OCULTAS + [K]
assert [a * b + b for a, b in zip(tamanhos[:-1], tamanhos[1:])] == [l.count_params() for l in model.layers]
for camada in model.layers:
    W_, b_ = camada.get_weights()
    print(f"{camada.name}: W {W_.shape}, b {b_.shape}")

In [ ]:
model.compile(loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              optimizer=keras.optimizers.Adam(learning_rate=TAXA_APRENDIZADO),
              metrics=["accuracy"])
callbacks = ([keras.callbacks.EarlyStopping(monitor="val_loss", patience=PACIENCIA, restore_best_weights=True)]
             if PACIENCIA else [])
historico = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCAS,
                      batch_size=TAMANHO_LOTE, callbacks=callbacks, verbose=0)
n_lotes = int(np.ceil(len(X_train) / TAMANHO_LOTE))
print(f"épocas: {len(historico.history['loss'])} | lotes por época: {n_lotes} (cada época percorre o treino inteiro)")

plt.figure(figsize=(6, 3.5))
plt.plot(historico.history["loss"], label="treino")
plt.plot(historico.history["val_loss"], label="validação")
plt.xlabel("época"); plt.ylabel("perda (entropia cruzada)"); plt.legend(); plt.title("Curvas de perda")
plt.show()

## 6. Previsão e avaliação

In [ ]:
# Um exemplo: logits → probabilidades (softmax) → classe (argmax)
logits_1 = model.predict(X_test[:1], verbose=0)
prob_1 = my_softmax(logits_1)
print("logits:", logits_1.round(2))
print("probabilidades:", prob_1.round(3), "| soma:", round(float(prob_1.sum()), 3))
print("classe prevista:", classes[np.argmax(prob_1)], "| real:", classes[y_test[0]])

logits = model.predict(X_test, verbose=0)
proba = my_softmax(logits)
assert np.allclose(proba, keras.ops.convert_to_numpy(keras.ops.softmax(logits)), atol=1e-5)
yhat = np.argmax(proba, axis=1)          # argmax dos logits dá o mesmo resultado
display(pd.Series({"acurácia": accuracy_score(y_test, yhat), "F1 macro": f1_score(y_test, yhat, average="macro"),
                   "log loss": log_loss(y_test, proba, labels=range(K))}, name="teste").round(4).to_frame().T)
print(classification_report(y_test, yhat, target_names=[str(c) for c in classes], digits=3, zero_division=0))

In [ ]:
fig, ax = plt.subplots(figsize=(1.1 * K + 2, 1.1 * K + 1.5))
ConfusionMatrixDisplay.from_predictions(codificador.inverse_transform(y_test), codificador.inverse_transform(yhat),
                                        ax=ax, colorbar=False, xticks_rotation=45)
ax.set_title("Matriz de confusão (teste)")
plt.tight_layout(); plt.show()

In [ ]:
# Erros do modelo no teste (generaliza o display_errors do original).
erros = np.flatnonzero(yhat != y_test)
print(f"{len(erros)} erros em {len(y_test)} exemplos de teste")
if FORMATO_IMAGEM is not None:
    X_img = X_te_df.to_numpy()
    grade_imagens(X_img, [f"{classes[a]},{classes[b]}" for a, b in zip(y_test, yhat)], titulo="rótulo, previsão")
    if len(erros):
        grade_imagens(X_img[erros], [f"{classes[y_test[i]]},{classes[yhat[i]]}" for i in erros],
                      n=min(16, len(erros)), titulo="erros: rótulo, previsão")
else:
    display(X_te_df.iloc[erros[:8], :8].assign(real=classes[y_test[erros[:8]]], previsto=classes[yhat[erros[:8]]],
                                                confianca=proba[erros[:8]].max(axis=1).round(3)))

## 7. Previsão para novos dados

In [ ]:
novos = X_df.sample(3, random_state=SEMENTE)    # >>> ADAPTAR: DataFrame com as mesmas colunas de `features`
p_novos = my_softmax(model.predict(prep.transform(novos).astype("float32"), verbose=0))
display(novos.iloc[:, :8].assign(previsao=classes[p_novos.argmax(axis=1)], confianca=p_novos.max(axis=1).round(3)))
model.save("mlp_multiclasse.keras")

## 8. Próximos passos (segundo o cartão do sistema)
- Métricas: F1 (macro, se as classes têm pesos iguais), log loss e a matriz de confusão por classe. Classes raras: lotes balanceados ou `class_weight` no `fit`.
- Mais épocas reduzem os erros no treino, mas só a curva de validação diz se isso generaliza.
- Em dados tabulares de porte médio, **prefira boosting**, a menos que a rede mostre ganho consistente em validação cruzada.